# 10. Evaluation Notebook

This notebook demonstrates the comprehensive evaluation framework for the Cirq-RAG-Code-Assistant.

## Components Covered
1. **Metrics Collection** - Collect code quality and agent performance metrics
2. **Benchmark Suite** - Run standard algorithm benchmarks
3. **Ablation Studies** - Evaluate impact of system components
4. **Report Generation** - Generate evaluation reports

## 1. Setup and Initialization

In [1]:
import sys
import os
from pathlib import Path
import cirq

# Add project root to path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Change working directory to project root for correct path resolution
os.chdir(project_root)
print(f"Working directory: {os.getcwd()}")

# Core imports
from src.cirq_rag_code_assistant.config import get_config, setup_logging
from src.cirq_rag_code_assistant.config.logging import setup_default_logging

# Evaluation imports
from src.evaluation.metrics import MetricsCollector
from src.evaluation.benchmark import BenchmarkSuite
from src.evaluation.ablation import AblationStudy
from src.evaluation.reports import ReportGenerator

# RAG and Agent imports
from src.rag.embeddings import EmbeddingModel
from src.rag.vector_store import VectorStore
from src.rag.knowledge_base import KnowledgeBase
from src.rag.retriever import Retriever
from src.rag.generator import Generator
from src.agents.designer import DesignerAgent
from src.agents.optimizer import OptimizerAgent
from src.agents.validator import ValidatorAgent
from src.agents.educational import EducationalAgent
from src.orchestration.orchestrator import Orchestrator
from src.tools.analyzer import CircuitAnalyzer

# Setup logging
setup_default_logging()
print("✅ All evaluation modules imported successfully!")

Working directory: D:\University\Uni\Semester 7\Generative AI\Project\Cirq-RAG-Code-Assistant


2025-12-07 17:04:31 | INFO     | src.cirq_rag_code_assistant.config.logging:setup_all:138 | Logging configuration completed


✅ All evaluation modules imported successfully!


## 2. Initialize System Components

In [2]:
# Initialize embedding model and vector store
print("Initializing embedding model...")
embedding_model = EmbeddingModel()
vector_store = VectorStore(embedding_model.get_embedding_dimension())

# Initialize knowledge base
print("Initializing knowledge base...")
knowledge_base = KnowledgeBase(
    embedding_model=embedding_model,
    vector_store=vector_store
)

# Load knowledge base
try:
    knowledge_base.load_from_directory()
    knowledge_base.load_index()
    print(f"✅ Loaded {len(knowledge_base.entries)} knowledge base entries")
except Exception as e:
    print(f"⚠️ Could not load knowledge base: {e}")
    print("Run 03_vector_store.ipynb first to initialize the knowledge base.")

2025-12-07 17:04:31 | INFO     | config.config_loader:load:93 | ✅ Loaded configuration from D:\University\Uni\Semester 7\Generative AI\Project\Cirq-RAG-Code-Assistant\config\config.json
2025-12-07 17:04:31 | INFO     | src.rag.embeddings:__init__:97 | Loading embedding model: BAAI/bge-base-en-v1.5
2025-12-07 17:04:31 | INFO     | src.rag.embeddings:__init__:98 | Using device: cpu


Initializing embedding model...
2025-12-07 17:04:31,811 - sentence_transformers.SentenceTransformer - INFO - SentenceTransformer.py:227 - Load pretrained SentenceTransformer: BAAI/bge-base-en-v1.5


2025-12-07 17:04:36 | INFO     | src.rag.embeddings:__init__:106 | ✅ Embedding model loaded successfully
2025-12-07 17:04:36 | INFO     | src.rag.embeddings:__init__:113 | Embedding dimension: 768
2025-12-07 17:04:36 | INFO     | src.rag.vector_store:_init_faiss:139 | Initialized FAISS index
2025-12-07 17:04:36 | INFO     | src.rag.vector_store:__init__:120 | Initialized VectorStore with faiss backend
2025-12-07 17:04:36 | INFO     | src.rag.vector_store:__init__:121 | Embedding dimension: 768
2025-12-07 17:04:36 | INFO     | src.rag.knowledge_base:__init__:100 | Initialized KnowledgeBase with 0 entries
2025-12-07 17:04:36 | INFO     | src.rag.knowledge_base:load_from_jsonl:116 | Loading knowledge base from data\knowledge_base\curated_designer_examples.jsonl
2025-12-07 17:04:36 | INFO     | src.rag.knowledge_base:load_from_jsonl:129 | Loaded 107 entries from data\knowledge_base\curated_designer_examples.jsonl
2025-12-07 17:04:36 | INFO     | src.rag.knowledge_base:load_from_jsonl:116 |

Initializing knowledge base...
✅ Loaded 140 knowledge base entries


In [3]:
# Initialize RAG components
print("Initializing RAG components...")
retriever = Retriever(knowledge_base)
generator = Generator(retriever)

# Initialize agents
print("Initializing agents...")
designer = DesignerAgent(retriever, generator)
optimizer = OptimizerAgent(retriever=retriever)
validator = ValidatorAgent(retriever=retriever)
educational = EducationalAgent(retriever)

# Create orchestrator
print("Creating orchestrator...")
orchestrator = Orchestrator(
    designer=designer,
    optimizer=optimizer,
    validator=validator,
    educational=educational
)

print("✅ All system components initialized!")

2025-12-07 17:04:36 | INFO     | src.rag.retriever:__init__:170 | Initialized Retriever with top_k=5, threshold=0.7, topic_boost=0.15, hybrid_scoring=True
2025-12-07 17:04:36 | INFO     | src.rag.generator:__init__:170 | Initialized Generator with ollama/cirq-designer-agent
2025-12-07 17:04:36 | INFO     | src.agents.base_agent:__init__:78 | Initialized DesignerAgent agent
2025-12-07 17:04:36 | INFO     | src.agents.base_agent:__init__:78 | Initialized OptimizerAgent agent
2025-12-07 17:04:36 | INFO     | src.rag.generator:__init__:170 | Initialized Generator with ollama/cirq-optimizer-agent
2025-12-07 17:04:36 | INFO     | src.agents.optimizer:__init__:74 | OptimizerAgent initialized (RAG: enabled)
2025-12-07 17:04:36 | INFO     | src.agents.base_agent:__init__:78 | Initialized ValidatorAgent agent
2025-12-07 17:04:36 | INFO     | src.tools.simulator:__init__:82 | Initialized simulator simulator
2025-12-07 17:04:36 | INFO     | src.agents.validator:__init__:94 | ValidatorAgent initial

Initializing RAG components...
Initializing agents...
Creating orchestrator...
✅ All system components initialized!


## 3. Metrics Collection

The `MetricsCollector` gathers code quality and agent performance metrics.

In [4]:
# Initialize metrics collector
metrics_collector = MetricsCollector()

# Create a sample circuit for analysis
q0, q1 = cirq.LineQubit.range(2)
sample_circuit = cirq.Circuit(
    cirq.H(q0),
    cirq.CNOT(q0, q1),
    cirq.measure(q0, q1, key='result')
)

print("Sample Bell State Circuit:")
print(sample_circuit)

2025-12-07 17:04:36 | INFO     | src.evaluation.metrics:__init__:20 | Initialized MetricsCollector


Sample Bell State Circuit:
0: ───H───@───M('result')───
          │   │
1: ───────X───M─────────────


In [5]:
# Analyze circuit using CircuitAnalyzer
analyzer = CircuitAnalyzer()
analysis = analyzer.analyze(sample_circuit)

print("\n📊 Circuit Analysis:")
print("=" * 40)
for key, value in analysis['metrics'].items():
    print(f"  {key}: {value}")


📊 Circuit Analysis:
  num_qubits: 2
  depth: 3
  num_operations: 3
  num_moments: 3
  num_measurements: 1


In [6]:
# Collect code quality metrics
sample_code = str(sample_circuit)
validation_result = {
    "validation_passed": True,
    "compilation": {"success": True}
}

quality_metrics = metrics_collector.collect_code_quality(sample_code, validation_result)

print("\n📈 Code Quality Metrics:")
print("=" * 40)
for key, value in quality_metrics.items():
    print(f"  {key}: {value}")


📈 Code Quality Metrics:
  code_length: 73
  num_lines: 3
  syntax_valid: True
  compilation_success: True


In [7]:
# Simulate agent performance metrics
agent_stats = {
    "designer": {"total_requests": 10, "successful_requests": 9, "failed_requests": 1},
    "optimizer": {"total_requests": 8, "successful_requests": 8, "failed_requests": 0},
    "validator": {"total_requests": 10, "successful_requests": 10, "failed_requests": 0},
    "educational": {"total_requests": 5, "successful_requests": 5, "failed_requests": 0},
}

print("\n🤖 Agent Performance Metrics:")
print("=" * 40)
for agent_name, stats in agent_stats.items():
    perf_metrics = metrics_collector.collect_agent_performance(agent_name, stats)
    print(f"\n  {agent_name.upper()}:")
    print(f"    Success Rate: {perf_metrics['success_rate']:.1%}")
    print(f"    Total Requests: {perf_metrics['total_requests']}")


🤖 Agent Performance Metrics:

  DESIGNER:
    Success Rate: 90.0%
    Total Requests: 10

  OPTIMIZER:
    Success Rate: 100.0%
    Total Requests: 8

  VALIDATOR:
    Success Rate: 100.0%
    Total Requests: 10

  EDUCATIONAL:
    Success Rate: 100.0%
    Total Requests: 5


In [8]:
# Get aggregated metrics
aggregated = metrics_collector.get_aggregated_metrics()

print("\n📊 Aggregated Metrics:")
print("=" * 40)
if "code_quality" in aggregated:
    print("\nCode Quality:")
    for key, value in aggregated["code_quality"].items():
        if isinstance(value, float):
            print(f"  {key}: {value:.2f}")
        else:
            print(f"  {key}: {value}")

if "agent_performance" in aggregated:
    print("\nAgent Performance:")
    for key, value in aggregated["agent_performance"].items():
        if isinstance(value, float):
            print(f"  {key}: {value:.2f}")
        else:
            print(f"  {key}: {value}")


📊 Aggregated Metrics:

Code Quality:
  total_samples: 1
  avg_code_length: 73.00
  syntax_valid_rate: 1.00
  compilation_success_rate: 1.00

Agent Performance:
  total_agents: 4
  avg_success_rate: 0.97


## 4. Benchmark Suite

Run standard benchmarks against common quantum algorithms.

In [9]:
# Initialize benchmark suite
benchmark_suite = BenchmarkSuite(
    orchestrator=orchestrator,
    metrics_collector=MetricsCollector()  # Fresh collector for benchmarks
)

print("📋 Standard Benchmark Test Cases:")
print("=" * 50)
for i, case in enumerate(BenchmarkSuite.STANDARD_BENCHMARKS, 1):
    print(f"  {i}. {case['query']} [{case['algorithm']}]")

2025-12-07 17:04:36 | INFO     | src.evaluation.metrics:__init__:20 | Initialized MetricsCollector
2025-12-07 17:04:36 | INFO     | src.evaluation.benchmark:__init__:41 | Initialized BenchmarkSuite


📋 Standard Benchmark Test Cases:
  1. Create a 2-qubit Bell state circuit [teleportation]
  2. Implement a 3-qubit Grover search algorithm [grover]
  3. Create a simple VQE circuit for 2 qubits [vqe]
  4. Implement a 2-qubit QAOA circuit [qaoa]


In [10]:
# Run benchmarks (this may take a while as it calls the LLM)
print("\n🚀 Running Benchmark Suite...")
print("(This may take several minutes as it processes each test case)\n")

try:
    benchmark_results = benchmark_suite.run_benchmarks()
    
    print("\n" + "=" * 50)
    print("📊 BENCHMARK RESULTS")
    print("=" * 50)
    print(f"Total Tests: {benchmark_results['total_tests']}")
    print(f"Passed: {benchmark_results['passed']}")
    print(f"Failed: {benchmark_results['failed']}")
    print(f"Pass Rate: {benchmark_results['pass_rate']:.1%}")
    
    print("\n📝 Individual Test Results:")
    print("-" * 50)
    for result in benchmark_results['test_results']:
        status = "✅" if result['success'] and result['validation_passed'] else "❌"
        print(f"  {status} Test {result['test_id']}: {result['query'][:40]}...")
        if result.get('errors'):
            for error in result['errors']:
                print(f"      Error: {error}")

except Exception as e:
    print(f"❌ Benchmark error: {e}")
    print("\nNote: Ensure the knowledge base is loaded and Ollama is running.")
    benchmark_results = None

2025-12-07 17:04:36 | INFO     | src.evaluation.benchmark:run_benchmarks:65 | Running 4 benchmark tests...
2025-12-07 17:04:36 | INFO     | src.evaluation.benchmark:run_benchmarks:71 | Running benchmark 1/4: Create a 2-qubit Bell state circuit...
2025-12-07 17:04:36 | INFO     | src.orchestration.orchestrator:generate_code:127 | 🎨 [Stage 1/4] Designer Agent generating code...
2025-12-07 17:04:36 | INFO     | src.rag.generator:generate:193 | Retrieving context for query: Create a 2-qubit Bell state circuit...
2025-12-07 17:04:36 | INFO     | src.rag.generator:generate:235 | Generating code using ollama/cirq-designer-agent



🚀 Running Benchmark Suite...
(This may take several minutes as it processes each test case)



2025-12-07 17:04:49 | INFO     | src.rag.generator:generate:290 | ✅ Code generation completed
2025-12-07 17:04:49 | INFO     | src.orchestration.orchestrator:generate_code:141 | ✅ Designer generated code (924 chars)
2025-12-07 17:04:49 | INFO     | src.orchestration.orchestrator:generate_code:145 | ✅ [Stage 2/4] Validator Agent checking code...
2025-12-07 17:04:49 | INFO     | src.agents.validator:execute:121 | Validating code (attempt 1/3)...
2025-12-07 17:04:49 | INFO     | src.tools.simulator:simulate:147 | Simulation completed. Histogram: {'00': 534, '11': 490}
2025-12-07 17:04:49 | INFO     | src.agents.validator:execute:155 | ✅ Validation passed on attempt 1
2025-12-07 17:04:49 | INFO     | src.orchestration.orchestrator:generate_code:166 | ⚡ [Stage 3/4] Optimizer Agent improving code...
2025-12-07 17:04:49 | INFO     | src.orchestration.orchestrator:generate_code:172 |    Optimization iteration 1/3
2025-12-07 17:04:49 | INFO     | src.agents.optimizer:execute:330 | Retrieved 3 o


📊 BENCHMARK RESULTS
Total Tests: 4
Passed: 4
Failed: 0
Pass Rate: 100.0%

📝 Individual Test Results:
--------------------------------------------------
  ✅ Test 1: Create a 2-qubit Bell state circuit...
  ✅ Test 2: Implement a 3-qubit Grover search algori...
  ✅ Test 3: Create a simple VQE circuit for 2 qubits...
  ✅ Test 4: Implement a 2-qubit QAOA circuit...


## 5. Ablation Studies

Evaluate the impact of different system components by selectively disabling them.

In [11]:
# Define ablation test cases (smaller set for faster execution)
ablation_cases = [
    {"query": "Create a Bell state", "algorithm": "teleportation"},
    {"query": "Create a simple GHZ state for 3 qubits", "algorithm": "ghz"},
]

# Initialize ablation study
ablation_study = AblationStudy(benchmark_cases=ablation_cases)

print("🔬 Ablation Study Configuration:")
print("=" * 50)
print(f"Test Cases: {len(ablation_cases)}")
for case in ablation_cases:
    print(f"  - {case['query']}")

2025-12-07 17:05:20 | INFO     | src.rag.embeddings:__init__:97 | Loading embedding model: BAAI/bge-base-en-v1.5
2025-12-07 17:05:20 | INFO     | src.rag.embeddings:__init__:98 | Using device: cpu


2025-12-07 17:05:20,642 - sentence_transformers.SentenceTransformer - INFO - SentenceTransformer.py:227 - Load pretrained SentenceTransformer: BAAI/bge-base-en-v1.5


2025-12-07 17:05:24 | INFO     | src.rag.embeddings:__init__:106 | ✅ Embedding model loaded successfully
2025-12-07 17:05:24 | INFO     | src.rag.embeddings:__init__:113 | Embedding dimension: 768
2025-12-07 17:05:24 | INFO     | src.rag.vector_store:_init_faiss:139 | Initialized FAISS index
2025-12-07 17:05:24 | INFO     | src.rag.vector_store:__init__:120 | Initialized VectorStore with faiss backend
2025-12-07 17:05:24 | INFO     | src.rag.vector_store:__init__:121 | Embedding dimension: 768
2025-12-07 17:05:24 | INFO     | src.rag.knowledge_base:__init__:100 | Initialized KnowledgeBase with 0 entries
2025-12-07 17:05:24 | INFO     | src.rag.retriever:__init__:170 | Initialized Retriever with top_k=5, threshold=0.7, topic_boost=0.15, hybrid_scoring=True
2025-12-07 17:05:24 | INFO     | src.rag.generator:__init__:170 | Initialized Generator with ollama/cirq-designer-agent
2025-12-07 17:05:24 | INFO     | src.evaluation.ablation:__init__:41 | Initialized AblationStudy with 2 cases


🔬 Ablation Study Configuration:
Test Cases: 2
  - Create a Bell state
  - Create a simple GHZ state for 3 qubits


In [12]:
# Run ablation study with different variants
variants = ["full", "no_optimizer"]

print("\n🚀 Running Ablation Study...")
print(f"Variants to test: {variants}")
print("(This may take several minutes)\n")

try:
    ablation_results = ablation_study.run_study(variants)
    
    print("\n" + "=" * 50)
    print("📊 ABLATION STUDY RESULTS")
    print("=" * 50)
    
    for variant, results in ablation_results.items():
        print(f"\n🔹 Variant: {variant.upper()}")
        print("-" * 40)
        print(f"  Success Rate: {results['success_rate']:.1%}")
        print(f"  Avg Latency: {results['avg_latency']:.2f}s")
        print(f"  Total Cases: {results['total_cases']}")
        
except Exception as e:
    print(f"❌ Ablation study error: {e}")
    ablation_results = None

2025-12-07 17:05:24 | INFO     | src.evaluation.ablation:run_study:54 | Running ablation variant: full
2025-12-07 17:05:24 | INFO     | src.agents.base_agent:__init__:78 | Initialized DesignerAgent agent
2025-12-07 17:05:24 | INFO     | src.agents.base_agent:__init__:78 | Initialized OptimizerAgent agent
2025-12-07 17:05:24 | WARNING  | src.agents.optimizer:__init__:54 | OptimizerAgent initialized without retriever - RAG guidance disabled
2025-12-07 17:05:24 | INFO     | src.rag.generator:__init__:170 | Initialized Generator with ollama/cirq-optimizer-agent
2025-12-07 17:05:24 | INFO     | src.agents.optimizer:__init__:74 | OptimizerAgent initialized (RAG: disabled)
2025-12-07 17:05:24 | INFO     | src.agents.base_agent:__init__:78 | Initialized ValidatorAgent agent
2025-12-07 17:05:24 | WARNING  | src.agents.validator:__init__:72 | ValidatorAgent initialized without retriever - semantic validation disabled
2025-12-07 17:05:24 | INFO     | src.tools.simulator:__init__:82 | Initialized 


🚀 Running Ablation Study...
Variants to test: ['full', 'no_optimizer']
(This may take several minutes)



2025-12-07 17:05:33 | INFO     | src.rag.generator:generate:290 | ✅ Code generation completed
2025-12-07 17:05:33 | INFO     | src.orchestration.orchestrator:generate_code:141 | ✅ Designer generated code (693 chars)
2025-12-07 17:05:33 | INFO     | src.orchestration.orchestrator:generate_code:145 | ✅ [Stage 2/4] Validator Agent checking code...
2025-12-07 17:05:33 | INFO     | src.agents.validator:execute:121 | Validating code (attempt 1/3)...
2025-12-07 17:05:33 | INFO     | src.tools.simulator:simulate:147 | Simulation completed. Histogram: {'00': 483, '11': 541}
2025-12-07 17:05:33 | INFO     | src.agents.validator:execute:155 | ✅ Validation passed on attempt 1
2025-12-07 17:05:33 | INFO     | src.orchestration.orchestrator:generate_code:166 | ⚡ [Stage 3/4] Optimizer Agent improving code...
2025-12-07 17:05:33 | INFO     | src.orchestration.orchestrator:generate_code:172 |    Optimization iteration 1/3
2025-12-07 17:05:33 | INFO     | src.agents.validator:execute:121 | Validating co


📊 ABLATION STUDY RESULTS

🔹 Variant: FULL
----------------------------------------
  Success Rate: 50.0%
  Avg Latency: 17.06s
  Total Cases: 2

🔹 Variant: NO_OPTIMIZER
----------------------------------------
  Success Rate: 100.0%
  Avg Latency: 11.67s
  Total Cases: 2


## 6. Report Generation

Generate comprehensive evaluation reports.

In [13]:
# Initialize report generator
output_dir = Path("notebooks") / "outputs" / "reports"
report_generator = ReportGenerator(output_dir=output_dir)

print(f"📁 Report output directory: {output_dir}")

2025-12-07 17:06:21 | INFO     | src.evaluation.reports:__init__:33 | Initialized ReportGenerator with output dir: notebooks\outputs\reports


📁 Report output directory: notebooks\outputs\reports


In [14]:
# Generate JSON report
json_report_path = report_generator.generate_report(
    metrics=metrics_collector,
    benchmark_results=benchmark_results if benchmark_results else {},
    format="json"
)

print(f"✅ JSON Report generated: {json_report_path}")

2025-12-07 17:06:21 | INFO     | src.evaluation.reports:generate_report:71 | Generated report: notebooks\outputs\reports\evaluation_report_20251207_170621.json


✅ JSON Report generated: notebooks\outputs\reports\evaluation_report_20251207_170621.json


In [15]:
# Generate text report
text_report_path = report_generator.generate_report(
    metrics=metrics_collector,
    benchmark_results=benchmark_results if benchmark_results else {},
    format="text"
)

print(f"✅ Text Report generated: {text_report_path}")

# Display text report content
print("\n📄 Report Preview:")
print("=" * 60)
with open(text_report_path, 'r') as f:
    print(f.read())

2025-12-07 17:06:21 | INFO     | src.evaluation.reports:generate_report:71 | Generated report: notebooks\outputs\reports\evaluation_report_20251207_170621.text


✅ Text Report generated: notebooks\outputs\reports\evaluation_report_20251207_170621.text

📄 Report Preview:
Evaluation Report
Timestamp: 2025-12-07T17:06:21.940017

Metrics:
------------------------------------------------------------
  Code Quality:
    Total Samples: 1
    Avg Code Length: 73
    Syntax Valid Rate: 100.0%
    Compilation Success Rate: 100.0%

Benchmark Results:
------------------------------------------------------------
  Total Tests: 4
  Passed: 4
  Failed: 0
  Pass Rate: 100.0%


## 7. Summary

This notebook demonstrated the evaluation framework including:

- **MetricsCollector**: Gathered code quality and agent performance metrics
- **BenchmarkSuite**: Ran standard quantum algorithm benchmarks
- **AblationStudy**: Evaluated component impact through controlled experiments
- **ReportGenerator**: Created structured evaluation reports

The evaluation results can be used to:
1. Track system performance over time
2. Identify weak points in the pipeline
3. Compare different system configurations
4. Document system capabilities for stakeholders